# PULSO · Priorização de kits para manutenção
**Global Solution 2026.2 · FIAP · Engenharia de Software · Dynamic Programming**

Aluno: Cainã Sandes Batista · RM 568571 · Turma 2ESPR · Projeto: PULSO

Repositório: https://github.com/Sednas22/GS-CitizenTech

## Objetivos do PULSO
1. Promover maior autonomia de pessoas surdas, cegas e surdocegas durante a espera por atendimento presencial;
2. Reduzir o risco de chamadas não percebidas por meio de alertas sensoriais adaptados às preferências de cada pessoa;
3. Oferecer uma alternativa portátil e acessível de comunicação, sem depender de internet ou de equipamentos próprios dos estabelecimentos.

O PULSO é um dispositivo físico e **offline**: pulseira e transmissor ESP32 conversam por ESP-NOW. Este notebook **não** roda no dispositivo. É uma análise complementar, feita depois dos testes, para decidir **quais kits a manutenção deve olhar primeiro**.

**Execução no Google Colab:** abra este único arquivo `.ipynb` e use **Ambiente de execução > Executar tudo**. Não é necessário enviar CSVs nem arquivos `.py`; o próprio notebook contém o algoritmo, gera os dados simulados e executa os testes com bibliotecas padrão do Python.

## 1. Cenário e problema (o que este algoritmo resolve)
Para reduzir o risco de chamadas não percebidas, o PULSO depende de kits com alertas funcionais. Quando um kit apresenta falhas, a equipe precisa identificar quais merecem verificação primeiro. Com vários kits em uso, a equipe precisa de um critério claro para saber **qual kit atender primeiro**, sem tratar toda anomalia como defeito. Este notebook recebe os registros de testes e devolve uma **lista ordenada de kits**, cada um com o motivo da sua posição.

O ponto central é interpretar os registros com cuidado:
- uma chamada **ENVIADA e não confirmada não é defeito**: a pessoa pode simplesmente não ter respondido;
- uma falha em um canal que a pessoa **não usa** não prejudica ninguém;
- se **muitos kits falham no mesmo local**, o suspeito é o local, não os kits.

## 2. Fonte dos dados (integrada ao notebook)
**Dados 100% simulados**, sem pessoas reais ou testes reais dos dispositivos. O gerador abaixo usa semente fixa **2026** e produz automaticamente os mesmos 30 kits, 368 chamadas e 90 ensaios utilizados no relatório original. Não exige nenhum upload no Colab.

- `kits.csv`: identificação, preferências, bateria e status do kit;
- `chamadas.csv`: tentativas por kit, estabelecimento e status;
- `ensaios.csv`: resultado de verificação manual de cada canal.

Esses CSVs são **criados automaticamente ao executar o notebook**, apenas no ambiente de execução; não precisam acompanhar o arquivo da entrega. O ranking final também pode ser exportado em CSV.

## 3. Algoritmo de priorização (código completo incorporado)
O código abaixo é a implementação original de `priorizacao.py`, incorporada neste notebook. Usa apenas a biblioteca padrão do Python.

In [1]:
"""Priorização de kits PULSO para manutenção (Dynamic Programming, GS 2026.2).

O PULSO é um dispositivo físico offline (pulseira + transmissor ESP32). Este módulo
NÃO roda no dispositivo: é uma análise complementar, feita depois, sobre registros de
testes (simulados) para decidir quais kits devem ser olhados primeiro pela manutenção.

Ideia central
-------------
1. Agregar os registros por kit em uma única passagem (dicionários).
2. Para cada kit ATIVO, calcular três severidades entre 0 e 1:
   - canais de alerta: falhas nos canais que a pessoa realmente usa;
   - comunicação: falhas ACIMA do esperado para os locais onde o kit foi usado;
   - bateria: pulseira ou transmissor com pouca carga.
3. Combinar as severidades em uma pontuação de 0 a 100 e ordenar.

Regras de interpretação (decisões justificadas no relatório):
- Chamada ENVIADA sem confirmação NÃO é defeito: a pessoa pode não ter respondido.
  Ela só gera uma observação, nunca pontuação.
- Se todos os canais habilitados de um kit falharam no ensaio, ele não consegue alertar
  ninguém: nível CRITICA, sempre no topo.
- Se muitos kits falham em um mesmo local, o suspeito é o local, não os kits.

Complexidade: O(C + T + K log K), com C chamadas, T ensaios e K kits.
"""


import csv
import math
from dataclasses import dataclass
from pathlib import Path

CANAIS = ("vibracao", "luz", "audio")
STATUS_KIT = ("ATIVO", "MANUTENCAO", "DESABILITADO")
STATUS_CHAMADA = ("CONFIRMADA", "ENVIADA", "FALHA")
RESULTADOS_ENSAIO = ("OK", "FALHA")


# --------------------------------------------------------------------------- #
# Parâmetros e estruturas de dados
# --------------------------------------------------------------------------- #
@dataclass(frozen=True)
class Parametros:
    """Critérios ajustáveis da priorização (valores de partida, não verdades absolutas)."""

    peso_canal: float = 0.50          # sem alerta funcional a pessoa não é avisada
    peso_comunicacao: float = 0.35    # falhas de envio afetam todas as chamadas
    peso_bateria: float = 0.15        # bateria baixa se resolve com recarga
    min_chamadas_kit: int = 5         # abaixo disso a taxa do kit não é confiável
    min_chamadas_local: int = 10      # abaixo disso a taxa do local não é confiável
    taxa_falha_maxima: float = 0.30   # taxa acima do esperado que vale severidade 1
    bateria_critica: float = 10.0     # <= 10% : severidade 1
    bateria_ok: float = 30.0          # >= 30% : severidade 0
    limiar_alta: float = 30.0         # pontuação mínima do nível ALTA
    limiar_media: float = 15.0        # pontuação mínima do nível MEDIA
    limiar_sem_confirmacao: float = 0.50  # taxa que gera observação (sem pontuar)

    def __post_init__(self) -> None:
        soma = self.peso_canal + self.peso_comunicacao + self.peso_bateria
        if not math.isclose(soma, 1.0, abs_tol=1e-9):
            raise ValueError(f"Os pesos devem somar 1.0 (soma atual: {soma}).")
        if not 0 < self.taxa_falha_maxima <= 1:
            raise ValueError("taxa_falha_maxima deve estar em (0, 1].")
        if not 0 <= self.bateria_critica < self.bateria_ok <= 100:
            raise ValueError("Use 0 <= bateria_critica < bateria_ok <= 100.")
        if not 0 < self.limiar_media < self.limiar_alta <= 100:
            raise ValueError("Use 0 < limiar_media < limiar_alta <= 100.")
        if self.min_chamadas_kit < 1 or self.min_chamadas_local < 1:
            raise ValueError("Os mínimos de chamadas devem ser pelo menos 1.")


@dataclass(frozen=True)
class Kit:
    id_kit: int
    canais: frozenset          # canais de alerta habilitados pela pessoa
    bateria_pulseira: float
    bateria_transmissor: float
    status: str


@dataclass(frozen=True)
class Chamada:
    id_chamada: int
    id_kit: int
    id_estabelecimento: int
    status: str


@dataclass(frozen=True)
class Ensaio:
    id_kit: int
    canal: str
    resultado: str


@dataclass
class ResultadoKit:
    id_kit: int
    nivel: str
    score: float
    sem_alerta: bool
    s_canal: float
    s_comunicacao: float
    s_bateria: float
    n_chamadas: int
    taxa_falha_bruta: float
    taxa_falha_acima_esperado: float
    taxa_sem_confirmacao: float
    motivos: list
    observacoes: list


# --------------------------------------------------------------------------- #
# Leitura e validação dos CSV
# --------------------------------------------------------------------------- #
def _ler_csv(caminho: Path, colunas: tuple) -> list:
    """Lê um CSV e confere se tem exatamente as colunas esperadas."""
    with open(caminho, newline="", encoding="utf-8") as arquivo:
        leitor = csv.DictReader(arquivo)
        if leitor.fieldnames is None or set(leitor.fieldnames) != set(colunas):
            raise ValueError(
                f"{caminho.name}: colunas esperadas {list(colunas)}, "
                f"encontradas {leitor.fieldnames}."
            )
        return list(leitor)


def _inteiro(linha: dict, campo: str, origem: str, numero: int) -> int:
    try:
        return int(linha[campo])
    except (TypeError, ValueError):
        raise ValueError(f"{origem}, linha {numero}: '{campo}' deve ser inteiro.") from None


def _bateria(linha: dict, campo: str, origem: str, numero: int) -> float:
    try:
        valor = float(linha[campo])
    except (TypeError, ValueError):
        raise ValueError(f"{origem}, linha {numero}: '{campo}' deve ser numérico.") from None
    if not 0 <= valor <= 100:
        raise ValueError(f"{origem}, linha {numero}: '{campo}' deve estar entre 0 e 100.")
    return valor


def carregar_kits(caminho) -> list:
    caminho = Path(caminho)
    colunas = ("id_kit", "usa_vibracao", "usa_luz", "usa_audio",
               "bateria_pulseira", "bateria_transmissor", "status_kit")
    kits, vistos = [], set()
    for numero, linha in enumerate(_ler_csv(caminho, colunas), start=2):
        id_kit = _inteiro(linha, "id_kit", caminho.name, numero)
        if id_kit in vistos:
            raise ValueError(f"{caminho.name}, linha {numero}: id_kit {id_kit} repetido.")
        vistos.add(id_kit)
        flags = {}
        for canal in CANAIS:
            valor = linha[f"usa_{canal}"]
            if valor not in ("0", "1"):
                raise ValueError(f"{caminho.name}, linha {numero}: usa_{canal} deve ser 0 ou 1.")
            flags[canal] = valor == "1"
        if not any(flags.values()):
            raise ValueError(f"{caminho.name}, linha {numero}: ao menos um canal deve estar habilitado.")
        if linha["status_kit"] not in STATUS_KIT:
            raise ValueError(f"{caminho.name}, linha {numero}: status_kit inválido ({linha['status_kit']}).")
        kits.append(Kit(
            id_kit=id_kit,
            canais=frozenset(c for c, ligado in flags.items() if ligado),
            bateria_pulseira=_bateria(linha, "bateria_pulseira", caminho.name, numero),
            bateria_transmissor=_bateria(linha, "bateria_transmissor", caminho.name, numero),
            status=linha["status_kit"],
        ))
    return kits


def carregar_chamadas(caminho) -> list:
    caminho = Path(caminho)
    colunas = ("id_chamada", "id_kit", "id_estabelecimento", "status")
    chamadas, vistos = [], set()
    for numero, linha in enumerate(_ler_csv(caminho, colunas), start=2):
        id_chamada = _inteiro(linha, "id_chamada", caminho.name, numero)
        if id_chamada in vistos:
            raise ValueError(f"{caminho.name}, linha {numero}: id_chamada {id_chamada} repetido.")
        vistos.add(id_chamada)
        if linha["status"] not in STATUS_CHAMADA:
            raise ValueError(f"{caminho.name}, linha {numero}: status inválido ({linha['status']}).")
        chamadas.append(Chamada(
            id_chamada=id_chamada,
            id_kit=_inteiro(linha, "id_kit", caminho.name, numero),
            id_estabelecimento=_inteiro(linha, "id_estabelecimento", caminho.name, numero),
            status=linha["status"],
        ))
    return chamadas


def carregar_ensaios(caminho) -> list:
    caminho = Path(caminho)
    colunas = ("id_kit", "canal", "resultado")
    ensaios, vistos = [], set()
    for numero, linha in enumerate(_ler_csv(caminho, colunas), start=2):
        id_kit = _inteiro(linha, "id_kit", caminho.name, numero)
        if linha["canal"] not in CANAIS:
            raise ValueError(f"{caminho.name}, linha {numero}: canal inválido ({linha['canal']}).")
        if linha["resultado"] not in RESULTADOS_ENSAIO:
            raise ValueError(f"{caminho.name}, linha {numero}: resultado inválido ({linha['resultado']}).")
        if (id_kit, linha["canal"]) in vistos:
            raise ValueError(f"{caminho.name}, linha {numero}: ensaio repetido para kit {id_kit}, canal {linha['canal']}.")
        vistos.add((id_kit, linha["canal"]))
        ensaios.append(Ensaio(id_kit, linha["canal"], linha["resultado"]))
    return ensaios


# --------------------------------------------------------------------------- #
# Algoritmo
# --------------------------------------------------------------------------- #
def taxa_falha_por_local(chamadas: list, params: Parametros) -> dict:
    """Taxa de FALHA de cada estabelecimento (todas as chamadas, de todos os kits).

    Locais com poucas chamadas recebem taxa 0: sem amostra, não há desconto.
    """
    totais: dict = {}
    falhas: dict = {}
    for c in chamadas:
        totais[c.id_estabelecimento] = totais.get(c.id_estabelecimento, 0) + 1
        if c.status == "FALHA":
            falhas[c.id_estabelecimento] = falhas.get(c.id_estabelecimento, 0) + 1
    return {
        local: (falhas.get(local, 0) / total if total >= params.min_chamadas_local else 0.0)
        for local, total in totais.items()
    }


def severidade_bateria(nivel: float, params: Parametros) -> float:
    """1 até a bateria crítica, 0 a partir da bateria ok, linear entre as duas."""
    if nivel <= params.bateria_critica:
        return 1.0
    if nivel >= params.bateria_ok:
        return 0.0
    return (params.bateria_ok - nivel) / (params.bateria_ok - params.bateria_critica)


def _classificar(score: float, sem_alerta: bool, params: Parametros) -> str:
    if sem_alerta:
        return "CRITICA"
    if score >= params.limiar_alta:
        return "ALTA"
    if score >= params.limiar_media:
        return "MEDIA"
    if score > 0:
        return "BAIXA"
    return "SEM_PENDENCIA"


def avaliar_kits(kits: list, chamadas: list, ensaios: list,
                 params: Parametros = Parametros()) -> list:
    """Calcula a pontuação de cada kit ATIVO. Kits em manutenção ou desabilitados ficam de fora."""
    ids_kits = {k.id_kit for k in kits}
    for c in chamadas:
        if c.id_kit not in ids_kits:
            raise ValueError(f"Chamada {c.id_chamada} refere-se ao kit inexistente {c.id_kit}.")
    for e in ensaios:
        if e.id_kit not in ids_kits:
            raise ValueError(f"Ensaio refere-se ao kit inexistente {e.id_kit}.")

    taxa_local = taxa_falha_por_local(chamadas, params)

    # Uma passagem pelas chamadas: total, falhas, falhas esperadas pelo local, sem confirmação.
    agregado: dict = {}
    for c in chamadas:
        acc = agregado.setdefault(c.id_kit, [0, 0, 0.0, 0])
        acc[0] += 1
        acc[2] += taxa_local[c.id_estabelecimento]       # quanto o local "explica" de falha
        if c.status == "FALHA":
            acc[1] += 1
        elif c.status == "ENVIADA":
            acc[3] += 1

    ensaios_por_kit: dict = {}
    for e in ensaios:
        ensaios_por_kit.setdefault(e.id_kit, {})[e.canal] = e.resultado

    resultados = []
    for kit in kits:
        if kit.status != "ATIVO":
            continue
        total, falhas, esperado, sem_conf = agregado.get(kit.id_kit, [0, 0, 0.0, 0])
        motivos, observacoes = [], []

        # 1) Canais de alerta: só contam os canais habilitados pela pessoa.
        ensaio_kit = ensaios_por_kit.get(kit.id_kit, {})
        habilitados = sorted(kit.canais)
        com_falha = [c for c in habilitados if ensaio_kit.get(c) == "FALHA"]
        sem_ensaio = [c for c in habilitados if c not in ensaio_kit]
        s_canal = len(com_falha) / len(habilitados)
        sem_alerta = len(com_falha) == len(habilitados)
        if com_falha:
            motivos.append(
                f"Canal(is) habilitado(s) com falha no ensaio: {', '.join(com_falha)} "
                f"({len(com_falha)} de {len(habilitados)})"
            )
        if sem_alerta:
            motivos.append("Nenhum canal de alerta funcional: o kit não consegue avisar a pessoa")
        if sem_ensaio:
            observacoes.append(f"CANAL_SEM_ENSAIO: {', '.join(sem_ensaio)}")

        # 2) Comunicação: só a parte das falhas que o local não explica.
        taxa_bruta = falhas / total if total else 0.0
        taxa_excesso = max(0.0, falhas - esperado) / total if total else 0.0
        taxa_sem_conf = sem_conf / total if total else 0.0
        if total < params.min_chamadas_kit:
            s_comunicacao = 0.0
            observacoes.append(
                f"AMOSTRA_INSUFICIENTE: {total} chamada(s); mínimo {params.min_chamadas_kit}"
            )
        else:
            s_comunicacao = min(1.0, taxa_excesso / params.taxa_falha_maxima)
            if s_comunicacao > 0:
                motivos.append(
                    f"Falhas de comunicação: {falhas} de {total} chamadas ({taxa_bruta:.0%}); "
                    f"{taxa_excesso:.0%} acima do esperado pelos locais"
                )
            if taxa_sem_conf >= params.limiar_sem_confirmacao:
                observacoes.append(
                    f"SEM_CONFIRMACAO_RECORRENTE: {sem_conf} de {total} chamadas ({taxa_sem_conf:.0%}); "
                    "não pontua, pode ser comportamento do usuário"
                )

        # 3) Bateria: vale o dispositivo em pior estado.
        s_bateria = max(severidade_bateria(kit.bateria_pulseira, params),
                        severidade_bateria(kit.bateria_transmissor, params))
        if s_bateria > 0:
            motivos.append(
                f"Bateria baixa: pulseira {kit.bateria_pulseira:.0f}%, "
                f"transmissor {kit.bateria_transmissor:.0f}%"
            )

        score = 100 * (params.peso_canal * s_canal
                       + params.peso_comunicacao * s_comunicacao
                       + params.peso_bateria * s_bateria)
        resultados.append(ResultadoKit(
            id_kit=kit.id_kit,
            nivel=_classificar(score, sem_alerta, params),
            score=score,
            sem_alerta=sem_alerta,
            s_canal=s_canal,
            s_comunicacao=s_comunicacao,
            s_bateria=s_bateria,
            n_chamadas=total,
            taxa_falha_bruta=taxa_bruta,
            taxa_falha_acima_esperado=taxa_excesso,
            taxa_sem_confirmacao=taxa_sem_conf,
            motivos=motivos,
            observacoes=observacoes,
        ))
    return resultados


def ordenar(resultados: list) -> list:
    """Kits sem alerta funcional primeiro; depois maior pontuação; empate pelo menor id."""
    return sorted(resultados, key=lambda r: (not r.sem_alerta, -r.score, r.id_kit))


def salvar_ranking(ranking: list, caminho) -> None:
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    cabecalho = ["posicao", "id_kit", "nivel", "score", "s_canal", "s_comunicacao", "s_bateria",
                 "n_chamadas", "taxa_falha_bruta", "taxa_falha_acima_esperado",
                 "taxa_sem_confirmacao", "motivos", "observacoes"]
    with open(caminho, "w", newline="", encoding="utf-8") as arquivo:
        escritor = csv.writer(arquivo)
        escritor.writerow(cabecalho)
        for posicao, r in enumerate(ranking, start=1):
            escritor.writerow([
                posicao, r.id_kit, r.nivel, f"{r.score:.1f}", f"{r.s_canal:.2f}",
                f"{r.s_comunicacao:.2f}", f"{r.s_bateria:.2f}", r.n_chamadas,
                f"{r.taxa_falha_bruta:.2f}", f"{r.taxa_falha_acima_esperado:.2f}",
                f"{r.taxa_sem_confirmacao:.2f}", " | ".join(r.motivos), " | ".join(r.observacoes),
            ])




### Gerador de dados simulados
O gerador original foi incorporado abaixo, usando exatamente as mesmas regras e a mesma semente.

In [2]:
"""Gera os dados SIMULADOS usados na análise de priorização (semente fixa = reproduzível).

ATENÇÃO: estes dados são inventados só para demonstrar o algoritmo. Não são resultados de
testes reais do PULSO e não contêm nenhum dado pessoal. Rodar este script de novo, com a
mesma semente, reproduz exatamente os mesmos três arquivos CSV.

Uso:  python dados/gerar_dados_simulados.py
"""

import csv
import random
from pathlib import Path

SEMENTE = 2026
N_KITS = 30

# Probabilidade de FALHA de comunicação por local (estabelecimento).
# O local 4 simula um ambiente ruim (por exemplo, muita interferência de rádio).
P_FALHA_LOCAL = {1: 0.03, 2: 0.04, 3: 0.03, 4: 0.45, 5: 0.05, 6: 0.03}
P_ENVIADA_PADRAO = 0.12  # chamada enviada e não confirmada (a pessoa pode não ter respondido)

# Casos plantados de propósito para que o algoritmo tenha o que encontrar e para testar as regras.
PREFERENCIAS_FIXAS = {7: (1, 0, 0), 26: (1, 1, 0), 14: (1, 1, 1), 22: (1, 1, 0)}  # vibração, luz, áudio
STATUS_FIXOS = {3: "MANUTENCAO", 9: "DESABILITADO", 27: "MANUTENCAO"}
BATERIAS_FIXAS = {18: (None, 8), 28: (6, None), 11: (22, None)}  # (pulseira, transmissor)
ENSAIOS_COM_FALHA = {(7, "vibracao"), (26, "vibracao"), (26, "luz"), (14, "luz"), (22, "audio")}
FALHA_EXTRA = {12: 0.38}      # kit 12: falha em qualquer local, além do esperado
P_ENVIADA_KIT = {21: 0.65}    # kit 21: quase nunca confirma, mas o envio funciona
P_FALHA_FIXA = {21: 0.0}      # kit 21: nenhuma falha de comunicação
LOCAL_PREFERIDO = {25: 4}     # kit 25: usado quase só no local ruim
N_CHAMADAS_FIXO = {15: 3}     # kit 15: poucas chamadas
SEQUENCIA_FIXA = {15: ["FALHA", "FALHA", "CONFIRMADA"]}


def gerar(pasta: Path) -> None:
    rng = random.Random(SEMENTE)
    padroes = [(1, 0, 0), (0, 1, 0), (0, 0, 1), (1, 1, 0), (1, 0, 1), (0, 1, 1), (1, 1, 1)]
    pesos = [3, 2, 1, 3, 2, 1, 4]

    kits, ensaios = [], []
    for id_kit in range(1, N_KITS + 1):
        prefs = PREFERENCIAS_FIXAS.get(id_kit) or rng.choices(padroes, pesos)[0]
        bat_p, bat_t = rng.randint(40, 100), rng.randint(40, 100)
        fixa_p, fixa_t = BATERIAS_FIXAS.get(id_kit, (None, None))
        bat_p = fixa_p if fixa_p is not None else bat_p
        bat_t = fixa_t if fixa_t is not None else bat_t
        kits.append([id_kit, *prefs, bat_p, bat_t, STATUS_FIXOS.get(id_kit, "ATIVO")])
        for canal in ("vibracao", "luz", "audio"):
            falhou = (id_kit, canal) in ENSAIOS_COM_FALHA
            ensaios.append([id_kit, canal, "FALHA" if falhou else "OK"])

    chamadas, id_chamada = [], 1
    for id_kit in range(1, N_KITS + 1):
        n = N_CHAMADAS_FIXO.get(id_kit, rng.randint(8, 18))
        for i in range(n):
            if id_kit in LOCAL_PREFERIDO and rng.random() < 0.85:
                local = LOCAL_PREFERIDO[id_kit]
            elif id_kit in (12, 21):
                local = rng.choice([1, 2, 3, 5, 6])  # fora do local ruim, para não misturar causas
            else:
                local = rng.choice(list(P_FALHA_LOCAL))
            if id_kit in SEQUENCIA_FIXA:
                status = SEQUENCIA_FIXA[id_kit][i]
            else:
                p_falha = P_FALHA_FIXA.get(id_kit, min(0.95, P_FALHA_LOCAL[local] + FALHA_EXTRA.get(id_kit, 0)))
                p_enviada = P_ENVIADA_KIT.get(id_kit, P_ENVIADA_PADRAO)
                sorteio = rng.random()
                if sorteio < p_falha:
                    status = "FALHA"
                elif sorteio < p_falha + p_enviada:
                    status = "ENVIADA"
                else:
                    status = "CONFIRMADA"
            chamadas.append([id_chamada, id_kit, local, status])
            id_chamada += 1

    pasta.mkdir(parents=True, exist_ok=True)
    escrever(pasta / "kits.csv",
             ["id_kit", "usa_vibracao", "usa_luz", "usa_audio", "bateria_pulseira", "bateria_transmissor", "status_kit"], kits)
    escrever(pasta / "chamadas.csv", ["id_chamada", "id_kit", "id_estabelecimento", "status"], chamadas)
    escrever(pasta / "ensaios.csv", ["id_kit", "canal", "resultado"], ensaios)
    print(f"Gerados: {len(kits)} kits, {len(chamadas)} chamadas, {len(ensaios)} ensaios em {pasta}")


def escrever(caminho: Path, cabecalho: list, linhas: list) -> None:
    with open(caminho, "w", newline="", encoding="utf-8") as arquivo:
        escritor = csv.writer(arquivo)
        escritor.writerow(cabecalho)
        escritor.writerows(linhas)




In [3]:
# Gera, carrega e apresenta os dados sem importar arquivos do projeto
dados = Path("dados")
gerar(dados)

kits = carregar_kits(dados / "kits.csv")
chamadas = carregar_chamadas(dados / "chamadas.csv")
ensaios = carregar_ensaios(dados / "ensaios.csv")
print(f"{len(kits)} kits | {len(chamadas)} chamadas | {len(ensaios)} ensaios")

def mostrar_csv(nome, n=6):
    print(f"\n{nome} (primeiras linhas)")
    for linha in (dados / nome).read_text(encoding="utf-8").splitlines()[:n]:
        print("  " + linha)

for arquivo in ("kits.csv", "chamadas.csv", "ensaios.csv"):
    mostrar_csv(arquivo)

Gerados: 30 kits, 368 chamadas, 90 ensaios em dados
30 kits | 368 chamadas | 90 ensaios

kits.csv (primeiras linhas)
  id_kit,usa_vibracao,usa_luz,usa_audio,bateria_pulseira,bateria_transmissor,status_kit
  1,1,0,0,72,100,ATIVO
  2,1,1,0,95,100,ATIVO
  3,1,0,0,54,97,MANUTENCAO
  4,1,0,1,75,66,ATIVO
  5,1,1,1,75,93,ATIVO

chamadas.csv (primeiras linhas)
  id_chamada,id_kit,id_estabelecimento,status
  1,1,6,CONFIRMADA
  2,1,6,ENVIADA
  3,1,5,CONFIRMADA
  4,1,2,CONFIRMADA
  5,1,2,CONFIRMADA

ensaios.csv (primeiras linhas)
  id_kit,canal,resultado
  1,vibracao,OK
  1,luz,OK
  1,audio,OK
  2,vibracao,OK
  2,luz,OK


## 4. Critérios da priorização
Cada kit ATIVO recebe três severidades entre 0 e 1, combinadas em uma pontuação de 0 a 100:

| Critério | Peso | Como é medido |
|---|---|---|
| Canais de alerta | 50% | fração dos canais **habilitados** pela pessoa que falharam no ensaio |
| Comunicação | 35% | taxa de `FALHA` **acima do esperado** para os locais em que o kit foi usado (taxa de 30% acima do esperado = severidade 1) |
| Bateria | 15% | pior entre pulseira e transmissor: severidade 1 até 10% de carga, 0 a partir de 30%, linear no meio |

Regras adicionais: kit com **todos** os canais habilitados com falha é `CRITICA` e vai para o topo; `ENVIADA` sem confirmação **não pontua** (só gera observação se for muito frequente); kits com menos de 5 chamadas não têm taxa de comunicação calculada; kits em `MANUTENCAO` ou `DESABILITADO` ficam fora.

Níveis: `CRITICA` (sem alerta funcional), `ALTA` (≥ 30 pontos), `MEDIA` (≥ 15), `BAIXA` (> 0) e `SEM_PENDENCIA` (0). Os pesos e limiares são valores de partida, definidos em `Parametros`, e podem ser ajustados.

In [4]:
params = Parametros()
print(params)

Parametros(peso_canal=0.5, peso_comunicacao=0.35, peso_bateria=0.15, min_chamadas_kit=5, min_chamadas_local=10, taxa_falha_maxima=0.3, bateria_critica=10.0, bateria_ok=30.0, limiar_alta=30.0, limiar_media=15.0, limiar_sem_confirmacao=0.5)


### Falha esperada por local
A taxa de falha de cada estabelecimento (todos os kits somados) mostra quanto da falha de um kit o local já explica.

In [5]:
taxa = taxa_falha_por_local(chamadas, params)
totais = {}
for c in chamadas:
    totais[c.id_estabelecimento] = totais.get(c.id_estabelecimento, 0) + 1
for local in sorted(taxa):
    print(f"local {local}: {totais[local]:>3} chamadas | taxa de falha {taxa[local]:.0%}")

local 1:  57 chamadas | taxa de falha 9%
local 2:  63 chamadas | taxa de falha 6%
local 3:  64 chamadas | taxa de falha 6%
local 4:  61 chamadas | taxa de falha 51%
local 5:  74 chamadas | taxa de falha 8%
local 6:  49 chamadas | taxa de falha 4%


## 5. Execução: ranking de manutenção

In [6]:
ranking = ordenar(avaliar_kits(kits, chamadas, ensaios, params))
fora = [k.id_kit for k in kits if k.status != "ATIVO"]
print(f"Kits analisados: {len(ranking)} | fora da análise (manutenção/desabilitados): {fora}\n")

print(f"{'#':>2} {'kit':>4} {'nível':<14} {'score':>6}  motivos")
for posicao, r in enumerate(ranking[:10], start=1):
    print(f"{posicao:>2} {r.id_kit:>4} {r.nivel:<14} {r.score:>6.1f}  {'; '.join(r.motivos)}")

contagem = {}
for r in ranking:
    contagem[r.nivel] = contagem.get(r.nivel, 0) + 1
print("\nKits por nível:", contagem)

Kits analisados: 27 | fora da análise (manutenção/desabilitados): [3, 9, 27]

 #  kit nível           score  motivos
 1    7 CRITICA          50.0  Canal(is) habilitado(s) com falha no ensaio: vibracao (1 de 1); Nenhum canal de alerta funcional: o kit não consegue avisar a pessoa
 2   26 CRITICA          50.0  Canal(is) habilitado(s) com falha no ensaio: luz, vibracao (2 de 2); Nenhum canal de alerta funcional: o kit não consegue avisar a pessoa
 3   12 ALTA             35.0  Falhas de comunicação: 9 de 18 chamadas (50%); 43% acima do esperado pelos locais
 4   18 MEDIA            23.6  Falhas de comunicação: 2 de 14 chamadas (14%); 7% acima do esperado pelos locais; Bateria baixa: pulseira 96%, transmissor 8%
 5   14 MEDIA            16.7  Canal(is) habilitado(s) com falha no ensaio: luz (1 de 3)
 6    5 MEDIA            16.2  Falhas de comunicação: 3 de 10 chamadas (30%); 14% acima do esperado pelos locais
 7   28 MEDIA            15.0  Bateria baixa: pulseira 6%, transmissor 70%
 8 

## 6. Conferindo os casos que mais importam
Cada caso abaixo foi plantado nos dados para verificar uma regra. As linhas com `assert` falham se o comportamento mudar.

In [7]:
por_id = {r.id_kit: r for r in ranking}

r = por_id[7]
print(f"Kit 7  : só vibração habilitada e ela falhou no ensaio -> {r.nivel}, posição {ranking.index(r) + 1}")
assert r.nivel == "CRITICA"

r = por_id[25]
print(f"Kit 25 : {r.taxa_falha_bruta:.0%} de falhas, mas {r.taxa_falha_acima_esperado:.0%} acima do esperado "
      f"(usado quase só no local ruim) -> {r.nivel}")
assert r.score == 0

r = por_id[21]
print(f"Kit 21 : {r.taxa_sem_confirmacao:.0%} das chamadas sem confirmação, nenhuma falha -> {r.nivel}; "
      f"observação: {r.observacoes[0].split(':')[0]}")
assert r.score == 0 and r.observacoes

r = por_id[22]
print(f"Kit 22 : áudio falhou no ensaio, mas a pessoa não usa áudio -> {r.nivel}")
assert r.s_canal == 0

r = por_id[15]
print(f"Kit 15 : {r.n_chamadas} chamadas apenas -> {r.observacoes[0].split(':')[0]}, sem pontuar comunicação")
assert r.s_comunicacao == 0

for id_kit in (3, 9, 27):
    assert id_kit not in por_id
print("Kits 3, 9 e 27 : em manutenção ou desabilitados -> fora da análise")

Kit 7  : só vibração habilitada e ela falhou no ensaio -> CRITICA, posição 1
Kit 25 : 38% de falhas, mas 0% acima do esperado (usado quase só no local ruim) -> SEM_PENDENCIA
Kit 21 : 56% das chamadas sem confirmação, nenhuma falha -> SEM_PENDENCIA; observação: SEM_CONFIRMACAO_RECORRENTE
Kit 22 : áudio falhou no ensaio, mas a pessoa não usa áudio -> SEM_PENDENCIA
Kit 15 : 3 chamadas apenas -> AMOSTRA_INSUFICIENTE, sem pontuar comunicação
Kits 3, 9 e 27 : em manutenção ou desabilitados -> fora da análise


## 7. Exemplo de saída
O ranking completo é salvo em `saida/ranking_manutencao.csv`, que a equipe de manutenção pode abrir em uma planilha.

In [8]:
salvar_ranking(ranking, "saida/ranking_manutencao.csv")
for linha in Path("saida/ranking_manutencao.csv").read_text(encoding="utf-8").splitlines()[:4]:
    print(linha)

posicao,id_kit,nivel,score,s_canal,s_comunicacao,s_bateria,n_chamadas,taxa_falha_bruta,taxa_falha_acima_esperado,taxa_sem_confirmacao,motivos,observacoes
1,7,CRITICA,50.0,1.00,0.00,0.00,8,0.00,0.00,0.12,Canal(is) habilitado(s) com falha no ensaio: vibracao (1 de 1) | Nenhum canal de alerta funcional: o kit não consegue avisar a pessoa,
2,26,CRITICA,50.0,1.00,0.00,0.00,17,0.00,0.00,0.12,"Canal(is) habilitado(s) com falha no ensaio: luz, vibracao (2 de 2) | Nenhum canal de alerta funcional: o kit não consegue avisar a pessoa",
3,12,ALTA,35.0,0.00,1.00,0.00,18,0.50,0.43,0.06,Falhas de comunicação: 9 de 18 chamadas (50%); 43% acima do esperado pelos locais,


## 8. Estrutura de dados e complexidade
- **Dicionários (`dict`)** guardam os contadores por kit e por local. Acessar e atualizar uma chave custa O(1) em média, então cada chamada é lida **uma única vez**.
- **Ordenação com `sorted`** usa uma chave composta `(não está sem alerta, -score, id)`. O desempate pelo menor `id` torna o resultado sempre igual para a mesma entrada.

Com C chamadas, T ensaios e K kits:

| Etapa | Custo |
|---|---|
| Gerar, ler e validar os CSV | O(C + T + K) |
| Taxa de falha por local | O(C) |
| Agregar chamadas por kit | O(C) |
| Pontuar cada kit | O(K) |
| Ordenar | O(K log K) |
| **Total** | **O(C + T + K log K)** |

A memória é O(K + L), com L locais. Abaixo, o mesmo algoritmo com dados replicados em escala crescente: o tempo deve crescer de forma aproximadamente linear.

In [9]:
import time

def replicar(vezes):
    kits_g, chamadas_g, ensaios_g = [], [], []
    for v in range(vezes):
        deslocamento = v * 1000
        for k in kits:
            kits_g.append(Kit(k.id_kit + deslocamento, k.canais, k.bateria_pulseira, k.bateria_transmissor, k.status))
        for i, c in enumerate(chamadas):
            chamadas_g.append(Chamada(c.id_chamada + v * 100000, c.id_kit + deslocamento, c.id_estabelecimento, c.status))
        for e in ensaios:
            ensaios_g.append(Ensaio(e.id_kit + deslocamento, e.canal, e.resultado))
    return kits_g, chamadas_g, ensaios_g

print(f"{'kits':>8} {'chamadas':>10} {'tempo (s)':>10}")
for vezes in (1, 10, 100, 500):
    k_g, c_g, e_g = replicar(vezes)
    inicio = time.perf_counter()
    ordenar(avaliar_kits(k_g, c_g, e_g, params))
    print(f"{len(k_g):>8} {len(c_g):>10} {time.perf_counter() - inicio:>10.3f}")

    kits   chamadas  tempo (s)
      30        368      0.002
     300       3680      0.007
    3000      36800      0.051
   15000     184000      0.219


## 9. Testes automatizados
Os **30 testes originais** também estão incorporados a este arquivo. Eles verificam as regras de canais, comunicação, bateria, ordenação, leitura dos dados e os casos simulados. O resultado esperado é `OK`.

In [10]:
"""Testes de priorizacao.py. Rodar com:  python -m unittest -v"""

import csv
import tempfile
import unittest
from pathlib import Path


PASTA_DADOS = Path("dados")


def kit(id_kit=1, canais=("vibracao", "luz", "audio"), bat_p=80, bat_t=80, status="ATIVO"):
    return Kit(id_kit, frozenset(canais), bat_p, bat_t, status)


def chamadas_do_kit(id_kit, local, status, quantidade, inicio=1):
    return [Chamada(inicio + i, id_kit, local, status) for i in range(quantidade)]


def ensaios_ok(id_kit, falhas=()):
    return [Ensaio(id_kit, c, "FALHA" if c in falhas else "OK") for c in CANAIS]


def avaliar_um(k, chamadas=(), ensaios=None, params=Parametros()):
    ensaios = ensaios_ok(k.id_kit) if ensaios is None else ensaios
    return avaliar_kits([k], list(chamadas), ensaios, params)[0]


class TestParametros(unittest.TestCase):
    def test_pesos_devem_somar_um(self):
        with self.assertRaises(ValueError):
            Parametros(peso_canal=0.5, peso_comunicacao=0.5, peso_bateria=0.5)

    def test_limiares_em_ordem(self):
        with self.assertRaises(ValueError):
            Parametros(limiar_alta=10, limiar_media=20)


class TestBateria(unittest.TestCase):
    def test_pontos_de_corte(self):
        params = Parametros()
        self.assertEqual(severidade_bateria(5, params), 1.0)
        self.assertEqual(severidade_bateria(10, params), 1.0)
        self.assertEqual(severidade_bateria(20, params), 0.5)
        self.assertEqual(severidade_bateria(30, params), 0.0)
        self.assertEqual(severidade_bateria(100, params), 0.0)


class TestCanais(unittest.TestCase):
    def test_sem_alerta_funcional_e_critico(self):
        r = avaliar_um(kit(canais=("vibracao",)), ensaios=ensaios_ok(1, falhas=("vibracao",)))
        self.assertTrue(r.sem_alerta)
        self.assertEqual(r.nivel, "CRITICA")
        self.assertEqual(r.s_canal, 1.0)

    def test_falha_em_canal_nao_habilitado_nao_conta(self):
        r = avaliar_um(kit(canais=("vibracao", "luz")), ensaios=ensaios_ok(1, falhas=("audio",)))
        self.assertEqual(r.s_canal, 0.0)
        self.assertEqual(r.nivel, "SEM_PENDENCIA")

    def test_falha_parcial_nao_e_critica(self):
        r = avaliar_um(kit(), ensaios=ensaios_ok(1, falhas=("luz",)))
        self.assertFalse(r.sem_alerta)
        self.assertAlmostEqual(r.s_canal, 1 / 3)
        self.assertEqual(r.nivel, "MEDIA")

    def test_canal_habilitado_sem_ensaio_gera_observacao(self):
        r = avaliar_um(kit(canais=("vibracao", "luz")), ensaios=[Ensaio(1, "vibracao", "OK")])
        self.assertTrue(any("CANAL_SEM_ENSAIO" in o for o in r.observacoes))
        self.assertEqual(r.s_canal, 0.0)


class TestComunicacao(unittest.TestCase):
    def test_falha_acima_do_esperado_pontua(self):
        # Local 1 com 4 falhas em 10 chamadas do kit 1 e 20 chamadas boas de outro kit (taxa do local 13%).
        c = (chamadas_do_kit(1, 1, "FALHA", 4, 1) + chamadas_do_kit(1, 1, "CONFIRMADA", 6, 10)
             + chamadas_do_kit(99, 1, "CONFIRMADA", 20, 1000))
        kits = [kit(1), kit(99)]
        ensaios = ensaios_ok(1) + ensaios_ok(99)
        r = {x.id_kit: x for x in avaliar_kits(kits, c, ensaios)}[1]
        self.assertGreater(r.s_comunicacao, 0)
        self.assertGreater(r.score, 0)

    def test_nao_confirmacao_nao_e_defeito(self):
        r = avaliar_um(kit(), chamadas_do_kit(1, 1, "ENVIADA", 10))
        self.assertEqual(r.score, 0.0)
        self.assertEqual(r.nivel, "SEM_PENDENCIA")
        self.assertTrue(any("SEM_CONFIRMACAO_RECORRENTE" in o for o in r.observacoes))

    def test_amostra_pequena_nao_pontua(self):
        r = avaliar_um(kit(), chamadas_do_kit(1, 1, "FALHA", 3))
        self.assertEqual(r.s_comunicacao, 0.0)
        self.assertTrue(any("AMOSTRA_INSUFICIENTE" in o for o in r.observacoes))

    def test_local_ruim_explica_as_falhas(self):
        # Local 2: 10 chamadas do kit 1 (5 falhas) + 10 do kit 2 (5 falhas) = 50% de falha no local.
        local_ruim = (chamadas_do_kit(1, 2, "FALHA", 5, 1) + chamadas_do_kit(1, 2, "CONFIRMADA", 5, 10)
                      + chamadas_do_kit(2, 2, "FALHA", 5, 100) + chamadas_do_kit(2, 2, "CONFIRMADA", 5, 110))
        # Local 1: só o kit 3, com as mesmas 5 falhas em 10 chamadas, e nenhum outro kit falhando.
        local_normal = (chamadas_do_kit(3, 1, "FALHA", 5, 200) + chamadas_do_kit(3, 1, "CONFIRMADA", 5, 210)
                        + chamadas_do_kit(4, 1, "CONFIRMADA", 20, 300))
        kits = [kit(1), kit(2), kit(3), kit(4)]
        ensaios = [e for k in kits for e in ensaios_ok(k.id_kit)]
        res = {r.id_kit: r for r in avaliar_kits(kits, local_ruim + local_normal, ensaios)}
        self.assertEqual(res[1].taxa_falha_bruta, res[3].taxa_falha_bruta)  # mesma taxa bruta
        self.assertLess(res[1].score, res[3].score)                          # mas o local ruim "explica"
        self.assertEqual(res[1].s_comunicacao, 0.0)


class TestBateriaNoScore(unittest.TestCase):
    def test_vale_o_dispositivo_em_pior_estado(self):
        r = avaliar_um(kit(bat_p=90, bat_t=8))
        self.assertEqual(r.s_bateria, 1.0)
        self.assertAlmostEqual(r.score, 15.0)
        self.assertEqual(r.nivel, "MEDIA")


class TestAvaliacaoEOrdenacao(unittest.TestCase):
    def test_kits_fora_de_ativo_sao_ignorados(self):
        kits = [kit(1), kit(2, status="MANUTENCAO"), kit(3, status="DESABILITADO")]
        ensaios = [e for k in kits for e in ensaios_ok(k.id_kit)]
        res = avaliar_kits(kits, [], ensaios)
        self.assertEqual([r.id_kit for r in res], [1])

    def test_critico_vem_primeiro_mesmo_com_score_menor(self):
        kits = [kit(1, bat_t=5), kit(2, canais=("vibracao",))]
        ensaios = ensaios_ok(1) + ensaios_ok(2, falhas=("vibracao",))
        chamadas = chamadas_do_kit(1, 1, "FALHA", 10)
        ranking = ordenar(avaliar_kits(kits, chamadas, ensaios))
        self.assertEqual([r.id_kit for r in ranking], [2, 1])
        self.assertEqual(ranking[0].nivel, "CRITICA")

    def test_empate_resolve_pelo_menor_id(self):
        kits = [kit(5, bat_t=5), kit(2, bat_t=5)]
        ensaios = ensaios_ok(5) + ensaios_ok(2)
        ranking = ordenar(avaliar_kits(kits, [], ensaios))
        self.assertEqual([r.id_kit for r in ranking], [2, 5])

    def test_chamada_de_kit_inexistente(self):
        with self.assertRaises(ValueError):
            avaliar_kits([kit(1)], chamadas_do_kit(7, 1, "FALHA", 1), ensaios_ok(1))

    def test_entrada_vazia(self):
        self.assertEqual(avaliar_kits([], [], []), [])


class TestLeituraDeCsv(unittest.TestCase):
    def escrever(self, linhas, cabecalho):
        pasta = tempfile.TemporaryDirectory()
        self.addCleanup(pasta.cleanup)
        caminho = Path(pasta.name) / "arquivo.csv"
        with open(caminho, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(cabecalho)
            w.writerows(linhas)
        return caminho

    CAB_KITS = ["id_kit", "usa_vibracao", "usa_luz", "usa_audio",
                "bateria_pulseira", "bateria_transmissor", "status_kit"]

    def test_kits_validos(self):
        c = self.escrever([[1, 1, 0, 0, 80, 90, "ATIVO"]], self.CAB_KITS)
        kits = carregar_kits(c)
        self.assertEqual(kits[0].canais, frozenset({"vibracao"}))

    def test_exige_ao_menos_um_canal(self):
        c = self.escrever([[1, 0, 0, 0, 80, 90, "ATIVO"]], self.CAB_KITS)
        with self.assertRaises(ValueError):
            carregar_kits(c)

    def test_bateria_fora_do_intervalo(self):
        c = self.escrever([[1, 1, 0, 0, 120, 90, "ATIVO"]], self.CAB_KITS)
        with self.assertRaises(ValueError):
            carregar_kits(c)

    def test_status_invalido_e_id_repetido(self):
        c = self.escrever([[1, 1, 0, 0, 80, 90, "QUEBRADO"]], self.CAB_KITS)
        with self.assertRaises(ValueError):
            carregar_kits(c)
        c = self.escrever([[1, 1, 0, 0, 80, 90, "ATIVO"], [1, 0, 1, 0, 80, 90, "ATIVO"]], self.CAB_KITS)
        with self.assertRaises(ValueError):
            carregar_kits(c)

    def test_colunas_erradas(self):
        c = self.escrever([[1, "FALHA"]], ["id_chamada", "status"])
        with self.assertRaises(ValueError):
            carregar_chamadas(c)

    def test_status_de_chamada_invalido(self):
        c = self.escrever([[1, 1, 1, "TALVEZ"]], ["id_chamada", "id_kit", "id_estabelecimento", "status"])
        with self.assertRaises(ValueError):
            carregar_chamadas(c)

    def test_ensaio_repetido(self):
        c = self.escrever([[1, "luz", "OK"], [1, "luz", "FALHA"]], ["id_kit", "canal", "resultado"])
        with self.assertRaises(ValueError):
            carregar_ensaios(c)


class TestSaida(unittest.TestCase):
    def test_salvar_ranking(self):
        ranking = ordenar(avaliar_kits([kit(1, bat_t=5)], [], ensaios_ok(1)))
        with tempfile.TemporaryDirectory() as pasta:
            destino = Path(pasta) / "sub" / "ranking.csv"
            salvar_ranking(ranking, destino)
            with open(destino, encoding="utf-8") as arquivo:
                linhas = list(csv.DictReader(arquivo))
        self.assertEqual(linhas[0]["posicao"], "1")
        self.assertEqual(linhas[0]["nivel"], "MEDIA")
        self.assertEqual(linhas[0]["score"], "15.0")


class TestDadosSimulados(unittest.TestCase):
    """Confere se o algoritmo acha os casos plantados em dados/ (regressão)."""

    @classmethod
    def setUpClass(cls):
        kits = carregar_kits(PASTA_DADOS / "kits.csv")
        chamadas = carregar_chamadas(PASTA_DADOS / "chamadas.csv")
        ensaios = carregar_ensaios(PASTA_DADOS / "ensaios.csv")
        cls.ranking = ordenar(avaliar_kits(kits, chamadas, ensaios))
        cls.por_id = {r.id_kit: r for r in cls.ranking}

    def test_criticos_no_topo(self):
        self.assertEqual({r.id_kit for r in self.ranking[:2]}, {7, 26})
        self.assertTrue(all(r.nivel == "CRITICA" for r in self.ranking[:2]))

    def test_kit_com_falha_geral_vem_logo_depois(self):
        self.assertEqual(self.ranking[2].id_kit, 12)
        self.assertEqual(self.ranking[2].nivel, "ALTA")

    def test_casos_que_nao_devem_pontuar(self):
        for id_kit in (21, 22, 25, 15):
            self.assertEqual(self.por_id[id_kit].score, 0.0, f"kit {id_kit}")

    def test_kits_fora_da_analise(self):
        for id_kit in (3, 9, 27):
            self.assertNotIn(id_kit, self.por_id)

    def test_baterias_baixas(self):
        self.assertEqual(self.por_id[28].nivel, "MEDIA")
        self.assertEqual(self.por_id[18].s_bateria, 1.0)




# Executa os 30 testes originais sem depender de outro arquivo .py
import sys
suite = unittest.TestSuite()
for classe in [TestParametros, TestBateria, TestCanais, TestComunicacao, TestBateriaNoScore, TestAvaliacaoEOrdenacao, TestLeituraDeCsv, TestSaida, TestDadosSimulados]:
    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(classe))
resultado_testes = unittest.TextTestRunner(verbosity=1, stream=sys.stdout).run(suite)
assert resultado_testes.wasSuccessful(), "Algum teste falhou."


..............................
----------------------------------------------------------------------
Ran 30 tests in 0.038s

OK


## 10. Limitações
- Os dados são **simulados**. Os resultados mostram que o algoritmo funciona, não o desempenho real do PULSO;
- Pesos e limiares são escolhas justificáveis, não verdades: com dados reais, deveriam ser calibrados;
- A taxa esperada de um local inclui as falhas do próprio kit analisado; com poucos kits no mesmo local, o desconto pode ser exagerado;
- Os ensaios por canal pressupõem uma verificação manual; o PULSO não tem autoteste embarcado nesta versão;
- A priorização indica **onde olhar primeiro**. A decisão de consertar ou desabilitar um kit continua sendo da equipe.

### Arquivos de saída no Colab
Depois de executar tudo, o ranking estará em `saida/ranking_manutencao.csv`. No painel esquerdo do Colab, abra **Arquivos** para acessar ou baixar o CSV, se desejar. Esse arquivo é gerado pelo notebook e **não precisa ser enviado separadamente**.

**Sobre o relatório:** a análise continua sendo uma proposta complementar para manutenção de kits, não uma função implementada no ESP32.